# Single-network, single-knob seizure model — NEURON

A single-compartment **HH + A-current (`kA`) + dynamic [K+]o (`kdyn`)** clustered
network with AMPA+NMDA synapses and two-timescale spike-frequency adaptation.

**One fixed network. Normal vs seizure differ by a SINGLE parameter:**
`sahp_ainc_slow` — the slow-AHP / M-current (Kv7/KCNQ) strength.

| state | `sahp_ainc_slow` | phenotype |
|---|---|---|
| **normal**  | 0.009 (strong adaptation) | quiet, sparse loose bursts (~0.2 Hz), [K+]o ~4 mM |
| **seizure** | 0.003 (weak adaptation)   | active loose bursts (~0.7 Hz, ≥35% participation), [K+]o ~4 mM |

Both states use `states.normal_state()` (so `tau_k` stays 200) — this is an
**adaptation-deficit (KCNQ/Kv7) seizure**, the mild-[K+]o bursting phenotype, *not*
the K+-clearance (`tau_k`) ictal route. Raising `sahp_ainc_slow` models a Kv7
opener (retigabine); lowering it models KCNQ2/3 loss-of-function.

This notebook: build one network → normal vs seizure rasters + loose-burst check
→ sweep the single knob → generate two datasets from the SAME wiring → run CCG +
learned-LIF inference on both, reporting connectivity AUC/FDR incl. inhibitory-edge AUC.

> **Prerequisite:** compile the mechanisms once (`cd neuron_simulation && nrnivmodl mechanisms`).

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)

from neuron_simulation import (
    topology, build_network, run_simulation, states, analysis, plotting, workflows,
)

# ==========================================================================
# SINGLE-NETWORK, SINGLE-KNOB seizure model.
# ONE fixed network (seed 1). Normal vs seizure differ by a SINGLE parameter:
#   sahp_ainc_slow (slow-AHP / M-current strength).
#     NORMAL  = 0.009  strong adaptation  -> quiet, sparse loose bursts
#     SEIZURE = 0.003  weak adaptation    -> active loose bursts
# Both states use states.normal_state() (tau_k stays 200); adaptation-deficit
# (KCNQ/Kv7) seizure, NOT the K+-clearance (tau_k) route.
# ==========================================================================
SAHP_NORMAL, SAHP_SEIZURE = 0.009, 0.003   # the ONLY thing that differs between states

CONFIG = {
    'topology': dict(
        num_clusters=10, neurons_per_cluster_range=(4, 40),
        inhibitory_probability=0.2, space_size=15.0,
        cell_type_specific=False,
        within_cluster_prob=0.25, between_cluster_prob=0.06,
        ln_sigma=0.5, target_density=None, seed=1,
    ),
    'build': dict(
        synapse_model='ampa_nmda', exc_tau=5.0, tau_nmda=350.0, nmda_ratio=3.0,
        exc_weight_scale=3.0, inh_weight_scale=2.5, depression_d=0.2, tau_d=500.0,
        noise_rate=18.0, noise_weight=0.004, adapt=True,
        sahp_ainc_fast=0.005, sahp_tau_fast=300.0,
        sahp_ainc_slow=SAHP_NORMAL, sahp_tau_slow=6500.0, delay_per_distance=2.0,
    ),
    'sim': dict(dt=0.05, discard_transient_ms=1000.0),
}

def build_for(sahp_slow):
    """build_kwargs for one state: identical network, only sahp_ainc_slow differs."""
    b = dict(CONFIG['build']); b['sahp_ainc_slow'] = sahp_slow; return b

def loose_bursts(res, duration_ms, thresh=0.35):
    """Loose (low-participation) burst detector: the default >80% detector misses
    these events, so use >=35% participation."""
    b = analysis.detect_network_bursts(res['spike_data'], N, duration_ms,
                                       participation_threshold=thresh, burn_in_ms=0.0)
    return b, analysis.burst_statistics(b, duration_ms, burn_in_ms=0.0)

print('config ready (single-knob: SAHP_NORMAL=%.3f, SAHP_SEIZURE=%.3f)' % (SAHP_NORMAL, SAHP_SEIZURE))

## 1. Build the topology (log-normal — preferred)

Continuous heavy-tailed degree distribution (no bimodal gap) at ~3–4% density.
Use `topology.build_topology(...)` for the discrete-hub variant.

In [ ]:
topo = topology.build_topology_lognormal(**CONFIG['topology'])
N = topo['n_neurons']

# One-call topology view: prints the inter/between-cluster connection stats
# (density, within/between rates, ratio, hub share) and returns the 4-panel
# overview (spatial layout, hub fan-out, cluster-sorted connection matrix,
# out-degree distribution). Supersedes the separate map + degree plots.
stats, fig = plotting.topology_report(topo)
plt.show()

## 2. Normal state — strong adaptation (`sahp_ainc_slow = 0.009`)

Quiet, sparse **loose** bursts (~0.2 Hz) with a low-firing async baseline; [K+]o ~4 mM.
Loose bursts are low-participation, so we detect them at **≥35% participation**
(the default >80% detector misses them). Both rasters are shown; genuine
network-wide synchrony survives the row shuffle.

In [ ]:
DUR = 60000.0   # 60 s -> captures several multi-second inter-burst intervals
normal = workflows.run_single_state(
    topo, state=states.normal_state(), build_kwargs=build_for(SAHP_NORMAL),
    duration=DUR, record_ko=True, **CONFIG['sim'])
_, nstats = loose_bursts(normal, DUR)          # loose (>=35%) burst detector
nrate = analysis.firing_rate_summary(normal['spike_data'], DUR, burn_in_ms=0.0)['mean_rate_hz']
print('NORMAL  sahp_ainc_slow=%.3f' % SAHP_NORMAL)
print('  loose bursts: %d (%.2f Hz) | participation %.2f | firing %.2f Hz | [K+]o %.1f-%.1f mM' % (
    nstats['n_bursts'], nstats['burst_rate_hz'], nstats['mean_participation'], nrate,
    float(normal['ko_data']['mean_ko'].min()), float(normal['ko_data']['mean_ko'].max())))

for rr, tag in [(False, 'cluster-sorted'), (True, 'randomized rows')]:
    fig = plotting.plot_raster_with_ko(
        normal['spike_data'], N, DUR, normal['ko_data'],
        is_inhibitory=topo['neuron_is_inhibitory'],
        cluster_assignments=topo['cluster_assignments'],
        title='NORMAL (sahp_ainc_slow=%.3f, %s)' % (SAHP_NORMAL, tag), randomize_rows=rr)
    plt.show()

## 3. Seizure state — weak adaptation (`sahp_ainc_slow = 0.003`)

The **same wired network**, only the slow-AHP knob lowered. Active loose bursts
(~0.7 Hz, ≥35% participation); [K+]o still ~4 mM (adaptation-deficit, not a
K+-clearance ictal event).

In [ ]:
seizure = workflows.run_single_state(
    topo, state=states.normal_state(), build_kwargs=build_for(SAHP_SEIZURE),
    duration=DUR, record_ko=True, **CONFIG['sim'])
_, sstats = loose_bursts(seizure, DUR)
srate = analysis.firing_rate_summary(seizure['spike_data'], DUR, burn_in_ms=0.0)['mean_rate_hz']
print('SEIZURE sahp_ainc_slow=%.3f' % SAHP_SEIZURE)
print('  loose bursts: %d (%.2f Hz) | participation %.2f | firing %.2f Hz | [K+]o %.1f-%.1f mM' % (
    sstats['n_bursts'], sstats['burst_rate_hz'], sstats['mean_participation'], srate,
    float(seizure['ko_data']['mean_ko'].min()), float(seizure['ko_data']['mean_ko'].max())))
assert sstats['burst_rate_hz'] > nstats['burst_rate_hz'], 'seizure should burst more than normal'
assert sstats['mean_participation'] >= 0.35, 'seizure loose bursts should reach >=35% participation'
print('OK: lowering sahp_ainc_slow (0.009 -> 0.003) turns quiet -> active bursting on the SAME network')

for rr, tag in [(False, 'cluster-sorted'), (True, 'randomized rows')]:
    fig = plotting.plot_raster_with_ko(
        seizure['spike_data'], N, DUR, seizure['ko_data'],
        is_inhibitory=topo['neuron_is_inhibitory'],
        cluster_assignments=topo['cluster_assignments'],
        title='SEIZURE (sahp_ainc_slow=%.3f, %s)' % (SAHP_SEIZURE, tag), randomize_rows=rr)
    plt.show()

## 4. The single knob — `sahp_ainc_slow` sweep

`sahp_ainc_slow` is the whole story: sweeping it 0.003 → 0.009 gives a graded
transition from active bursting (KCNQ loss) to quiet (Kv7-opener / retigabine).

In [ ]:
# Graded transition: sweep the ONLY knob (seizure 0.003 -> normal 0.009).
sweep_vals = [0.003, 0.004, 0.005, 0.007, 0.009]
SWDUR = 30000.0
rates, parts = [], []
for v in sweep_vals:
    r = workflows.run_single_state(topo, state=states.normal_state(),
                                   build_kwargs=build_for(v), duration=SWDUR, record_ko=True, **CONFIG['sim'])
    _, st = loose_bursts(r, SWDUR)
    rates.append(st['burst_rate_hz']); parts.append(st['mean_participation'])
    print('sahp_ainc_slow=%.3f -> loose-burst rate %.2f Hz, participation %.2f' % (v, st['burst_rate_hz'], st['mean_participation']))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep_vals, rates, 'o-', color='#c0392b')
ax.set_xlabel('sahp_ainc_slow  (slow-AHP / M-current strength)')
ax.set_ylabel('loose-burst rate (Hz)')
ax.set_title('Single knob: adaptation strength sets burst density\n(left = seizure / KCNQ loss, right = normal / Kv7 opener)')
ax.invert_xaxis()   # seizure (low sahp) on the left
plt.show()

## 5. Two datasets from the SAME network (normal vs seizure knob)

Both datasets use `states.normal_state()` and the **same** `seed=1` topology, so
the ground-truth wiring is identical; they differ ONLY by `sahp_ainc_slow`. This
is the slow cell (2 × 10 × 60 s). Saved to `NEURON data/normal` and `NEURON data/seizure`.

In [ ]:
# Two datasets from the SAME wired network (seed 1 -> identical ground truth);
# they differ ONLY by sahp_ainc_slow, so inference sees the same graph under two
# activity regimes: sparse-async (normal) vs burst-dense (seizure).
meta_n, dir_n = workflows.generate_dataset(
    n_recordings=10, recording_duration=60000.0,
    topology_kind='lognormal', topology_kwargs=CONFIG['topology'],
    build_kwargs=build_for(SAHP_NORMAL), state=states.normal_state(),
    record_voltage=False, save_dir='NEURON data/normal', **CONFIG['sim'])
print('normal dataset  ->', dir_n)

meta_s, dir_s = workflows.generate_dataset(
    n_recordings=10, recording_duration=60000.0,
    topology_kind='lognormal', topology_kwargs=CONFIG['topology'],
    build_kwargs=build_for(SAHP_SEIZURE), state=states.normal_state(),
    record_voltage=False, save_dir='NEURON data/seizure', **CONFIG['sim'])
print('seizure dataset ->', dir_s)
print('normal  per-rec spikes:', [r['num_spikes'] for r in meta_n['recordings']])
print('seizure per-rec spikes:', [r['num_spikes'] for r in meta_s['recordings']])

## 6. Inference on both datasets (same wiring, two regimes)

CCG baseline + learned-LIF on each session. `adapter.run_inference` reports the
overall connectivity **AUC/FDR** and the **inhibitory-edge AUC** (positive class
I→E, I→I; baseline to beat ~0.475). Same ground truth, so any difference is the
activity regime (burst-dense seizure vs sparse-async normal).

In [ ]:
import adapter  # inference/adapter.py

def score(tag, session_dir):
    print('\n========== %s ==========' % tag)
    adapter.validate_session_format(session_dir)
    s = adapter.run_inference(session_dir, run_learned=True, run_ccg=True,
                              learned_params={'n_epochs': 20, 'K': 60})
    L = s.get('learned') or {}
    print('%s learned: AUC=%s FDR=%.3f | inhibitory-edge AUC=%s | excitatory-edge AUC=%s' % (
        tag, L.get('auc'), L.get('fdr', float('nan')), L.get('auc_inhibitory'), L.get('auc_excitatory')))
    return s

summ_n = score('NORMAL', dir_n)
summ_s = score('SEIZURE', dir_s)

print('\n=== SUMMARY (identical wiring, two regimes) ===')
for tag, s in [('normal', summ_n), ('seizure', summ_s)]:
    L = s.get('learned') or {}; C = s.get('ccg') or {}
    print('%-8s learned AUC %s (inh %s / exc %s) | CCG AUC %s' % (
        tag, L.get('auc'), L.get('auc_inhibitory'), L.get('auc_excitatory'), C.get('auc')))

## Notes & caveats

- **The seizure knob here is `sahp_ainc_slow`**, not `tau_k`. Lowering it
  (0.009 → 0.003) reduces slow adaptation (KCNQ2/3 loss-of-function), turning the
  same network from quiet to actively bursting. Both states keep `tau_k=200` and
  use `normal_state()`, so [K+]o stays ~4 mM (mild-[K+]o bursting phenotype).
- For the **K+-clearance ictal mechanism** instead (higher [K+]o ~18 mM), use the
  `seizure_state` / `tau_k` route — but first set the `states.py` `NORMAL_SAHP_*`
  constants to match this base (`NORMAL_SAHP_SLOW=0.009`, `NORMAL_SAHP_FAST=0.005`)
  so it reduces adaptation from the true normal values.
- **Loose bursts** are low-participation events, so the default >80% burst detector
  misses them — this notebook uses `participation_threshold=0.35`.
- The **ground truth is the exact wired graph** (`connections`), identical across
  the normal and seizure datasets (same `seed=1`), so the two inference runs
  isolate the effect of activity regime on an identical network.

## Parameter tuning guide

All knobs below are `build_kwargs` (passed to `build_network`) unless noted. The
defaults already give the realistic regime; change these to move the operating
point. Effects are coupled -- see the trade-offs at the end.

**Network dynamics**

| Want to change | Knob | Direction |
|---|---|---|
| Burst *rate* (slower/faster) | `sahp_ainc_slow`, `sahp_tau_slow` | higher -> slower bursts (longer IBI). Default 0.2 Hz. |
| *Spikes per burst* (thinner) | `sahp_ainc_fast` | higher -> fewer spikes/burst. Default ~2. |
| *Inter-burst background* firing | `noise_weight`, `noise_rate` | higher -> more background (fills gaps) |
| Burst *duration* | `nmda_ratio`, `delay_per_distance` | higher -> longer (reverberation / propagation) |
| Overall excitability | `exc_weight_scale`, `inh_weight_scale` | exc up / inh down -> more excitable |

**Topology** (`topology.build_topology_lognormal(...)` args)

| Want to change | Knob | Notes |
|---|---|---|
| Clustering strength | `within_cluster_prob` | *only works with* `target_density=None` |
| Overall density / in-degree | `target_density` | `None` = emerge from probs; a number = fixed density |
| Network size | `num_clusters`, `neurons_per_cluster_range` | e.g. (4, 40) for heterogeneous clusters |
| Spatial extent | `space_size` | scales the distance-dependent delays |

**Seizure** (`states.py` module constants)

| Want to change | Knob | Direction |
|---|---|---|
| Seizure firing/burst rate | `SEIZURE_SAHP_SLOW`, `SEIZURE_SAHP_FAST` | higher -> calmer (more adaptation retained) |
| Ictal [K+]o level | `SEIZURE_TAU_K` | higher -> more [K+]o accumulation |
| Per-run severity | `states.seizure_state(severity)` | higher severity -> higher `tau_k` -> stronger |

**Coupled trade-offs (can't optimize independently)**

- **Burst rate vs inter-burst firing:** a slower burst rate needs a stronger slow AHP, which also suppresses inter-burst firing. Slower bursts -> sparser background.
- **Spikes/burst vs burst duration:** fewer spikes/cell -> shorter bursts (duration is roughly spikes x intra-burst ISI), unless bursts propagate spatially.
- **Seizure firing vs ictal [K+]o:** firing *drives* [K+]o, so a calmer seizure (bounded firing) also gives a bounded [K+]o (~8 mM). For a higher [K+]o at bounded firing, raise `SEIZURE_TAU_K` / severity rather than cutting adaptation.
- **Uniform vs empty gaps:** uniform background needs strong enough independent noise; below a floor the recurrent network organizes gap firing into sparse mini-bursts.

**Backward compatibility:** the pre-tuning regime is still reachable via `build_network(synapse_model='depsyn', adapt=False, delay_per_distance=0.0, ...)` and `build_topology_lognormal(target_density=0.035, within_cluster_prob=0.55, ...)`.